# Offline Ablation — Stage 2: A1 GPT Stance Filtering / A2 Skip Stance Filtering

**Based directly on the original baseline `step2_gpt_top8000_and_stance_labeling` notebook.** The baseline `topic_definition`, `build_label_prompt`, `extract_json_from_text`, and GPT request are copied from that notebook. No topic retrieval is performed here.

- **A1:** classify the three random Top-8000 files with the original GPT stance prompt and confidence threshold, then export labeled and clean Pro/Anti data for Stage 3.
- **A2:** do **not** call GPT, do **not** infer or assign stance labels; copy the original topic-aware `top8000_before_gpt` inputs unchanged for Stage 3 without stance quotas.
- **Full, A3, A4:** not computed here.

**This notebook is Stage 2 only.** Final 300-record repositories are produced later by Stage 3 (diversity) and Stage 4 (refinement), except A1 may have fewer than 300 when relevant candidates are insufficient. Never send the 8,000-record files directly to online evaluation.


In [ ]:
!pip -q install openai pandas tqdm pyarrow

In [ ]:
import os, json, time, hashlib, shutil
from pathlib import Path
from getpass import getpass
import pandas as pd
from tqdm.auto import tqdm
from openai import AzureOpenAI


In [ ]:
# EDIT THESE PATHS to point to the files in your Google Drive.
A1_INPUT_DIR = Path('/content/A1_random_only_v1/full/top8000_variants')
OUTPUT_DIR = Path('/content/stage2')

A1_FILES = {
    'blm': 'blm_top8000_A1_no_topic_retrieval.csv',
    'guns': 'guns_top8000_A1_no_topic_retrieval.csv',
    'samesex': 'samesex_top8000_A1_no_topic_retrieval.csv',
}

TEXT_COL = 'comment_text'
BATCH_SIZE = 50                 # baseline value
CONFIDENCE_THRESHOLD = 0.60   # baseline value
RUN_MODE = 'FULL'              # 'TEST' or 'FULL'
TEST_BATCHES_PER_TOPIC = 2     # 100 real GPT labels per topic in TEST
RUN_A1_GPT = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert RUN_MODE in {'TEST','FULL'}
print('Mode:',RUN_MODE,'Output:',OUTPUT_DIR)


## Verify input identity and schema (no GPT calls)

In [ ]:
inputs = {}
for variant, root, mapping in [('A1',A1_INPUT_DIR,A1_FILES)]:
    for topic, filename in mapping.items():
        path = root / filename
        if not path.exists():
            raise FileNotFoundError(f'{variant}/{topic}: {path}. Edit input paths before running.')
        df = pd.read_csv(path,encoding='utf-8-sig',low_memory=False)
        if len(df)!=8000: raise ValueError(f'{variant}/{topic}: expected 8000, got {len(df)}')
        if TEXT_COL not in df: raise ValueError(f'{variant}/{topic}: missing {TEXT_COL}')
        if 'comment_id' not in df: raise ValueError(f'{variant}/{topic}: missing comment_id')
        if df.comment_id.astype(str).duplicated().any(): raise ValueError(f'{variant}/{topic}: duplicate comment_id')
        if variant=='A2' and 'stance_label' in df.columns: raise ValueError('A2 must use BEFORE GPT files')
        inputs[variant,topic]=df
        print(variant,topic,'rows',len(df),'columns',len(df.columns))


## Original baseline topic definitions and GPT labeling prompt (copied without modification)

In [ ]:
def topic_definition(topic):
    if topic == "blm":
        return """
Topic: Black Lives Matter.

A comment is relevant only if it discusses Black Lives Matter, racial justice, systemic racism, police brutality against Black people, George Floyd, racial inequality, or direct arguments about BLM.

pro = supports BLM or the racial justice/systemic racism framing.
anti = criticizes BLM, rejects the racial justice/systemic racism framing, calls BLM divisive/racist/useless/violent, or supports All Lives Matter as a rejection of BLM.
neutral_or_mixed = relevant to BLM but unclear, balanced, factual, or both sides.
irrelevant = not actually about BLM.
"""

    if topic == "guns":
        return """
Topic: Gun control.

A comment is relevant only if it discusses gun control, firearm laws, gun rights, the Second Amendment, background checks, assault weapons, NRA, gun bans, open carry, concealed carry, or gun ownership policy.

pro = supports stricter gun control, restrictions, bans, background checks, or reducing gun access.
anti = opposes gun control, supports gun rights, Second Amendment framing, firearm ownership, or argues gun restrictions are harmful/useless.
neutral_or_mixed = relevant to gun control but unclear, balanced, factual, or both sides.
irrelevant = mentions guns only incidentally or is not about gun policy.
"""

    if topic == "samesex":
        return """
Topic: Same-sex marriage legalization.

A comment is relevant only if it discusses same-sex marriage, gay marriage, marriage equality, civil unions, or legal recognition of same-sex couples.

pro = supports same-sex marriage, marriage equality, LGBT marriage rights, or equal legal recognition.
anti = opposes same-sex marriage, rejects marriage equality, or supports limiting marriage to heterosexual couples.
neutral_or_mixed = relevant to same-sex marriage but unclear, balanced, factual, or both sides.
irrelevant = about LGBT issues generally but not same-sex marriage, or not about this topic.
"""

    raise ValueError(topic)

In [ ]:
def build_label_prompt(topic, batch_df, text_col):
    rows = []

    for _, row in batch_df.iterrows():
        text = str(row[text_col]) if pd.notna(row[text_col]) else ""
        text = text.replace("\n", " ").strip()

        rows.append({
            "id": int(row["_label_local_id"]),
            "text": text[:1500],
        })

    return f"""
You are labeling political comments for a research dataset.

{topic_definition(topic)}

Classify each comment into exactly one label:
pro, anti, neutral_or_mixed, irrelevant.

Important:
First decide whether the comment is actually relevant to the topic.
If it is not clearly about the topic, label it irrelevant.
Do not infer relevance only because one keyword appears.
Do not classify based on toxicity or politeness.
Focus only on the political stance toward the topic.

Return valid JSON only in this exact format:
{{
  "items": [
    {{
      "id": 0,
      "stance_label": "pro",
      "confidence": 0.95,
      "short_reason": "brief explanation"
    }}
  ]
}}

Comments:
{json.dumps(rows, ensure_ascii=False)}
"""

In [ ]:
def extract_json_from_text(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise

## GPT client — original Azure deployment configuration

In [ ]:
from getpass import getpass
from openai import AzureOpenAI
if RUN_A1_GPT:
  AZURE_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT") or os.environ.get("ENDPOINT_URL")
  AZURE_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY")
  if not AZURE_ENDPOINT or not AZURE_API_KEY:
      raise EnvironmentError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY before running.")
  client = AzureOpenAI(
      azure_endpoint=AZURE_ENDPOINT,
      api_key=AZURE_API_KEY,
      api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
  )
  GPT_MODEL = os.environ.get("AZURE_OPENAI_DEPLOYMENT")

In [ ]:
def classify_batch(topic, batch_df, text_col, max_retries=4):
    prompt = build_label_prompt(topic, batch_df, text_col)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=AZURE_DEPLOYMENT_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a careful political stance classifier. Return valid JSON only."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
            )

            content = response.choices[0].message.content
            parsed = extract_json_from_text(content)

            return parsed.get("items", [])

        except Exception as e:
            print(f"Attempt {attempt + 1} failed:", repr(e))
            time.sleep(5 + attempt * 5)

    return []

## Safe per-batch checkpointing and validation

Each successfully validated 50-row GPT batch is stored on Drive. An invalid or incomplete GPT response raises an error rather than silently turning unclassified records into `irrelevant`. Re-run to resume. TEST and FULL have independent caches.

In [ ]:

import hashlib
import json
import os
import time
from pathlib import Path

LABELS = {"pro", "anti", "neutral_or_mixed", "irrelevant"}

CACHE_ROOT = Path(OUTPUT_DIR) / RUN_MODE / "A1" / "batches"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def input_hash(df):
    h = hashlib.sha256()

    for row in df[["comment_id", TEXT_COL]].itertuples(
        index=False, name=None
    ):
        h.update(
            json.dumps(
                [str(x) for x in row],
                ensure_ascii=False
            ).encode("utf-8")
        )
        h.update(b"\n")

    return h.hexdigest()


def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_name(path.name + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

    os.replace(tmp, path)


def run_a1_topic(topic):
    print("\n" + "=" * 80)
    print(f"Processing A1 topic: {topic}")
    print("=" * 80)

    # A1 already contains 8,000 randomly sampled records.
    # Do NOT apply select_top_relevant_rows again.
    df = inputs["A1", topic].reset_index(drop=True).copy()

    if TEXT_COL not in df.columns:
        raise ValueError(
            f"Column {TEXT_COL} not found. Columns: {df.columns.tolist()}"
        )

    print(f"Rows selected for GPT labeling: {len(df):,}")

    df["_label_local_id"] = range(len(df))

    limit = (
        min(len(df), BATCH_SIZE * TEST_BATCHES_PER_TOPIC)
        if RUN_MODE == "TEST"
        else len(df)
    )

    df = df.iloc[:limit].copy()

    # Hash the exact input used in this run.
    digest = input_hash(df)

    all_items = []

    for start in tqdm(
        range(0, len(df), BATCH_SIZE),
        desc=f"GPT labeling A1 {topic}"
    ):
        batch = df.iloc[start:start + BATCH_SIZE]

        checkpoint_path = (
            CACHE_ROOT / topic / f"batch_{start:05d}.json"
        )

        items = None

        # Resume: reuse an existing completed batch.
        if checkpoint_path.exists():
            with open(
                checkpoint_path, "r", encoding="utf-8"
            ) as f:
                checkpoint = json.load(f)

            if (
                checkpoint.get("input_sha256") == digest
                and checkpoint.get("start") == start
                and checkpoint.get("batch_size") == len(batch)
            ):
                items = checkpoint["items"]
            else:
                raise ValueError(
                    f"Checkpoint does not match current input: "
                    f"{checkpoint_path}"
                )

        # Same GPT call as the original baseline.
        if items is None:
            items = classify_batch(topic, batch, TEXT_COL)

            # Save the raw returned items without modifying them.
            write_json_atomic(
                checkpoint_path,
                {
                    "topic": topic,
                    "start": start,
                    "batch_size": len(batch),
                    "input_sha256": digest,
                    "items": items,
                }
            )

            time.sleep(0.5)

        # Same behavior as baseline: no extra validation or retries.
        all_items.extend(items)

    labels_df = pd.DataFrame(all_items)

    if labels_df.empty:
        labels_df = pd.DataFrame(
            columns=[
                "id",
                "stance_label",
                "confidence",
                "short_reason"
            ]
        )

    rename_map = {
        "stance": "stance_label",
        "label": "stance_label",
        "classification": "stance_label",
        "reason": "short_reason",
        "rationale": "short_reason",
    }

    labels_df = labels_df.rename(columns={
        old: new
        for old, new in rename_map.items()
        if old in labels_df.columns
    })

    for col, default in {
        "id": None,
        "stance_label": "irrelevant",
        "confidence": 0.0,
        "short_reason": "missing_or_parse_error",
    }.items():
        if col not in labels_df.columns:
            labels_df[col] = default

    labels_df["id"] = pd.to_numeric(
        labels_df["id"], errors="coerce"
    )
    labels_df["confidence"] = pd.to_numeric(
        labels_df["confidence"], errors="coerce"
    )

    labels_df = labels_df.dropna(subset=["id"])
    labels_df["id"] = labels_df["id"].astype(int)

    df_clean = df.drop(
        columns=[
            "id",
            "stance_label",
            "confidence",
            "short_reason",
            "stance",
            "label",
            "classification",
            "reason",
            "rationale",
        ],
        errors="ignore"
    )

    merged = df_clean.merge(
        labels_df[
            ["id", "stance_label", "confidence", "short_reason"]
        ],
        left_on="_label_local_id",
        right_on="id",
        how="left"
    )

    merged["stance_label"] = (
        merged["stance_label"].fillna("irrelevant")
    )

    merged["confidence"] = (
        merged["confidence"].fillna(0.0)
    )

    merged["short_reason"] = (
        merged["short_reason"].fillna("missing_or_parse_error")
    )

    allowed = {
        "pro",
        "anti",
        "neutral_or_mixed",
        "irrelevant"
    }

    merged.loc[
        ~merged["stance_label"].isin(allowed),
        "stance_label"
    ] = "irrelevant"

    dest = Path(OUTPUT_DIR) / RUN_MODE / "A1"
    dest.mkdir(parents=True, exist_ok=True)

    labeled_path = (
        dest / f"{topic}_top8000_all_labeled_A1.csv"
    )

    merged.to_csv(
        labeled_path,
        index=False,
        encoding="utf-8-sig"
    )

    clean_pro_anti = merged[
        (merged["stance_label"].isin(["pro", "anti"])) &
        (merged["confidence"] >= 0.60)
    ].copy()

    clean_path = (
        dest / f"{topic}_clean_pro_anti_A1.csv"
    )

    clean_pro_anti.to_csv(
        clean_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nSaved labeled file:", labeled_path)
    print("Saved clean pro/anti file:", clean_path)

    print("\nAll label counts:")
    print(merged["stance_label"].value_counts())

    print("\nClean pro/anti counts:")
    print(clean_pro_anti["stance_label"].value_counts())

    return merged, clean_pro_anti


## A1 — run original GPT stance labeling with resumable batches

In [ ]:
if RUN_A1_GPT:
    a1_results={}
    for topic in A1_FILES:
        a1_results[topic]=run_a1_topic(topic)


## Next steps

- **A1:** use `FULL/A1/*_clean_pro_anti_A1.csv` as input to the original stance-balanced Stage 3 diversity procedure, then original Stage 4 refinement. Do not assume 300 records will be available.
- **A2:** use `A2/for_stage3/*_A2_no_stance_for_diversity.csv` for a stance-agnostic diversity procedure selecting 400 total, then the explicitly modified six-criteria GPT refinement selecting the top 300 **without quotas**. These are *not* the original Step 3/4 without changes; they must be adapted carefully.
- **A3/A4:** start from their respective original baseline intermediates in later notebooks.
- **Online:** only after final repositories are produced, evaluate each variant on the same 600 test cases with the same online protocol as baseline.

**Pilot caution:** `TEST` saves only 100 labels/topic. Switching to `FULL` uses a separate cache and labels the full 8,000/topic. This avoids accidentally treating pilot outputs as complete.
